In [ ]:
import PIL
import torch
import torchvision
import numpy as np

torch.set_grad_enabled(False)

image_file = "image.png"

model = torchvision.models.detection.maskrcnn_resnet50_fpn(pretrained=True)
model = model.eval().cpu()

image = PIL.Image.open(image_file)
image_tensor = torchvision.transforms.functional.to_tensor(image).cpu()
output = model([image_tensor])[0]

coco_names = [
    "unlabeled",
    "person",
    "bicycle",
    "car",
    "motorcycle",
    "airplane",
    "bus",
    "train",
    "truck",
    "boat",
    "traffic light",
    "fire hydrant",
    "street sign",
    "stop sign",
    "parking meter",
    "bench",
    "bird",
    "cat",
    "dog",
    "horse",
    "sheep",
    "cow",
    "elephant",
    "bear",
    "zebra",
    "giraffe",
    "hat",
    "backpack",
    "umbrella",
    "shoe",
    "eye glasses",
    "handbag",
    "tie",
    "suitcase",
    "frisbee",
    "skis",
    "snowboard",
    "sports ball",
    "kite",
    "baseball bat",
    "baseball glove",
    "skateboard",
    "surfboard",
    "tennis racket",
    "bottle",
    "plate",
    "wine glass",
    "cup",
    "fork",
    "knife",
    "spoon",
    "bowl",
    "banana",
    "apple",
    "sandwich",
    "orange",
    "broccoli",
    "carrot",
    "hot dog",
    "pizza",
    "donut",
    "cake",
    "chair",
    "couch",
    "potted plant",
    "bed",
    "mirror",
    "dining table",
    "window",
    "desk",
    "toilet",
    "door",
    "tv",
    "laptop",
    "mouse",
    "remote",
    "keyboard",
    "cell phone",
    "microwave",
    "oven",
    "toaster",
    "sink",
    "refrigerator",
    "blender",
    "book",
    "clock",
    "vase",
    "scissors",
    "teddy bear",
    "hair drier",
    "toothbrush",
]

result = {"masks": [], "labels": [], "scores": [], "boxes": [], "areas": []}

H, W = np.array(image).shape[:2]
total_pixels = H * W

for i in range(len(output["scores"])):
    if output["scores"][i] <= 0.5:
        continue

    one_mask = output["masks"][i][0].cpu().numpy()
    one_mask[one_mask >= np.max(one_mask) * 0.5] = 1
    one_mask[one_mask < np.max(one_mask) * 0.5] = 0

    box = output["boxes"][i].int().cpu().numpy()
    score = float(output["scores"][i].cpu().numpy())
    label = int(output["labels"][i].cpu().numpy())
    area = int(np.sum(one_mask))

    result["masks"].append(one_mask)
    result["boxes"].append(box)
    result["scores"].append(score)
    result["labels"].append(label)
    result["areas"].append(area)

items = []
for i in range(len(result["scores"])):
    x1, y1, x2, y2 = result["boxes"][i]
    w = int(x2 - x1 + 1)
    h = int(y2 - y1 + 1)
    items.append(
        {
            "idx": i,
            "label": coco_names[result["labels"][i]],
            "score": result["scores"][i],
            "area": result["areas"][i],
            "share": result["areas"][i] / total_pixels,
            "x": int(x1),
            "y": int(y1),
            "x2": int(x2),
            "y2": int(y2),
            "w": w,
            "h": h,
        }
    )

items = sorted(items, key=lambda z: z["area"], reverse=True)

print("TOP OBJECTS SORTED BY MASK AREA:")
for obj in items[:15]:
    print(
        f"idx={obj['idx']}, "
        f"label={obj['label']}, "
        f"score={obj['score']:.6f}, "
        f"area={obj['area']}, "
        f"share={obj['share']:.6f}, "
        f"x={obj['x']}, y={obj['y']}, "
        f"x2={obj['x2']}, y2={obj['y2']}, "
        f"w={obj['w']}, h={obj['h']}"
    )

best_area = items[0]["area"]
print("\nALL OBJECTS WITH MAX AREA:")
for obj in items:
    if obj["area"] == best_area:
        print(
            f"idx={obj['idx']}, "
            f"label={obj['label']}, "
            f"score={obj['score']:.6f}, "
            f"area={obj['area']}, "
            f"share={obj['share']:.6f}, "
            f"x={obj['x']}, y={obj['y']}, "
            f"x2={obj['x2']}, y2={obj['y2']}, "
            f"w={obj['w']}, h={obj['h']}"
        )


TOP OBJECTS SORTED BY MASK AREA:
idx=1, label=airplane, score=0.988523, area=17273, share=0.063504, x=283, y=96, x2=628, y2=301, w=346, h=206
idx=19, label=truck, score=0.677819, area=4065, share=0.014945, x=388, y=385, x2=515, y2=421, w=128, h=37
idx=4, label=truck, score=0.964958, area=3551, share=0.013055, x=0, y=395, x2=156, y2=423, w=157, h=29
idx=17, label=airplane, score=0.689420, area=2752, share=0.010118, x=269, y=160, x2=455, y2=228, w=187, h=69
idx=7, label=truck, score=0.939616, area=2476, share=0.009103, x=343, y=275, x2=399, y2=325, w=57, h=51
idx=22, label=bus, score=0.606171, area=2238, share=0.008228, x=234, y=199, x2=327, y2=244, w=94, h=46
idx=15, label=airplane, score=0.746831, area=1438, share=0.005287, x=292, y=176, x2=450, y2=206, w=159, h=31
idx=10, label=airplane, score=0.897787, area=1291, share=0.004746, x=85, y=158, x2=197, y2=215, w=113, h=58
idx=16, label=airplane, score=0.717083, area=1174, share=0.004316, x=105, y=183, x2=197, y2=213, w=93, h=31
idx=24, 